# 🤖 Modelagem Preditiva, Validação Cruzada e Explicabilidade
### Desafio Técnico AI / MLOps Engineer — MadeinWeb
**Objetivo:** Construir o pipeline de Machine Learning, aplicar engenharia de features espaciais e socioeconômicas, realizar benchmark comparativo com 5-Fold Cross Validation e avaliar a explicabilidade com SHAP.

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

from src.data.loader import load_kc_houses, load_demographics, load_future_unseen
from src.features.pipeline import build_full_pipeline, HouseFeatureEngineer
from src.data.enricher import DemographicsEnricher

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Módulos carregados com sucesso!")

## 1. Carga dos Dados e Separação de Holdout Test (20%)
Garantimos a ausência total de *data leakage* separando 20% da base antes de qualquer ajuste de parâmetros ou transformações.

In [ ]:
df_houses = load_kc_houses(validate=True)
df_demo = load_demographics()

X = df_houses.drop(columns=['price'])
y = df_houses['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print(f"Treino: {X_train.shape[0]} amostras | Teste Holdout: {X_test.shape[0]} amostras")

## 2. Engenharia de Features: Espaciais, Temporais e Socioeconômicas
Criamos variáveis de domínio:
- Distância Haversine até os centros econômicos de **Seattle** e **Bellevue**.
- Idade do imóvel e tempo desde a última reforma (`base_year = 2015`).
- Relações estruturais: `living_to_lot_ratio`, `living_vs_neighbor_ratio`, `sqft_per_room`.
- Interações socioeconômicas: `affluence_score`, `living_x_zip_val`.

In [ ]:
engineer = HouseFeatureEngineer(base_year=2015)
enricher = DemographicsEnricher(demographics_df=df_demo).fit(X_train)

sample_features = engineer.transform(enricher.transform(X_train.head(5)))
print(f"Total de features geradas: {sample_features.shape[1]}")
print("Novas features geradas:", [c for c in sample_features.columns if c not in X_train.columns][:10])

## 3. Benchmark Comparativo com 5-Fold Cross Validation
Avaliamos 5 modelos distintos otimizando para métricas estatísticas e de negócio em dólares ($):
1. **Baseline**: Mediana
2. **Ridge**: Linear Regularizada
3. **Random Forest**: Ensemble de árvores
4. **XGBoost**: Gradient Boosting
5. **LightGBM**: Fast Gradient Boosting

In [ ]:
with open('../reports/model_benchmark_results.json', 'r') as f:
    benchmark_data = json.load(f)

df_benchmark = pd.DataFrame(benchmark_data).T
df_benchmark.columns = ['MAE ($ USD)', 'RMSE ($ USD)', 'MAPE (%)', 'R² Médio', 'Desvio R²']
print("=== TABELA COMPARATIVA DE BENCHMARK (5-FOLD CV) ===")
display(df_benchmark)

## 4. Visualização do Benchmark

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# R² Score
sns.barplot(x=df_benchmark.index, y=df_benchmark['R² Médio'], ax=axes[0], palette='Blues_r')
axes[0].set_title('R² Médio por Modelo (5-Fold CV)', fontweight='bold')
axes[0].set_ylim(0, 1.0)
axes[0].tick_params(axis='x', rotation=30)

# MAE
sns.barplot(x=df_benchmark.index, y=df_benchmark['MAE ($ USD)'], ax=axes[1], palette='Reds_r')
axes[1].set_title('Erro Médio Absoluto (MAE em US$)', fontweight='bold')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## 5. Avaliação do Modelo Campeão (LightGBM) no Conjunto de Teste (Holdout)

In [ ]:
pipeline = joblib.load('../models/model_pipeline.joblib')
y_test_pred = pipeline.predict(X_test)

r2_test = r2_score(y_test, y_test_pred)
mae_test = mean_absolute_error(y_test, y_test_pred)
rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))
mape_test = mean_absolute_percentage_error(y_test, y_test_pred) * 100.0

print(f"=== RESULTADOS NO CONJUNTO DE TESTE (HOLDOUT) ===")
print(f"R² Score:  {r2_test:.4f}")
print(f"MAE:       ${mae_test:,.2f}")
print(f"RMSE:      ${rmse_test:,.2f}")
print(f"MAPE:      {mape_test:.2f}%")

## 6. Gráficos de Resíduos e Dispersão: Real vs. Predito

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Dispersão
axes[0].scatter(y_test / 1000, y_test_pred / 1000, alpha=0.3, color='steelblue', s=15)
axes[0].plot([0, 4000], [0, 4000], 'r--', lw=2)
axes[0].set_title('Preço Real vs. Preço Predito (Milhares USD)', fontweight='bold')
axes[0].set_xlabel('Preço Real ($K)')
axes[0].set_ylabel('Preço Predito ($K)')
axes[0].set_xlim(0, 3500)
axes[0].set_ylim(0, 3500)

# Resíduos Percentuais
residuals_pct = ((y_test_pred - y_test) / y_test) * 100.0
sns.histplot(residuals_pct, bins=50, kde=True, ax=axes[1], color='darkcyan')
axes[1].set_title('Distribuição de Erros Residuais Percentuais', fontweight='bold')
axes[1].set_xlabel('Erro Residual (%)')
axes[1].set_xlim(-50, 50)

plt.tight_layout()
plt.show()

## 7. Explicabilidade do Modelo com SHAP (TreeExplainer)

In [ ]:
# Extração do pré-processamento e do estimador treinado
prep_step = pipeline.named_steps['prep']
model_estimator = pipeline.named_steps['model'].regressor_

X_test_prep = prep_step.transform(X_test.sample(300, random_state=42))
explainer = shap.TreeExplainer(model_estimator)
shap_values = explainer.shap_values(X_test_prep)

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test_prep, max_display=15, show=False)
plt.title('Importância Global das Features (SHAP Summary Plot)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Predições no Dataset de Teste Cego (`future_unseen_examples.csv`)

In [ ]:
df_predictions = pd.read_csv('../data/predictions/future_unseen_predictions.csv')
print(f"Total de predições geradas: {len(df_predictions)}")
display(df_predictions[['bedrooms', 'bathrooms', 'sqft_living', 'grade', 'zipcode', 'predicted_price_formatted']].head(10))

## 9. Conclusão e Próximos Passos
O modelo LightGBM atingiu **R² de 0.912** e **MAPE de 11.61%**, com erro médio de \$63.683 em um mercado de preço médio de \$540.000.
O pipeline serializado foi empacotado para servir via API FastAPI conteinerizada com Docker.